<div align="center">

# Olist Revenue Intelligence — Revenue Intelligence Analysis

### A decision-oriented analytical review of revenue structure, customer breadth, operational execution risk, and customer feedback signals based on the Olist analytical datasets.

</div>

## 1. Notebook Objectives

The purpose of this notebook is to transform the analytical datasets built in Notebook 2 into a structured revenue intelligence analysis.

The notebook focuses on four complementary questions:

- where revenue is concentrated
- how broad or concentrated the customer base is
- where operational execution appears weaker
- whether operational friction overlaps with weaker customer feedback signals

The analysis is built on two complementary analytical tables:

- **fact_order_items** for revenue and portfolio structure
- **fact_orders** for delivery KPIs, lateness analysis, and review-based feedback analysis

## 2. Load Analytical Tables

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

In [2]:
processed_path = "../data/processed/"

fact_order_items = pd.read_csv(processed_path + "fact_order_items.csv")
fact_orders = pd.read_csv(processed_path + "fact_orders.csv")

In [3]:
date_cols_items = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols_items:
    if col in fact_order_items.columns:
        fact_order_items[col] = pd.to_datetime(fact_order_items[col], errors="coerce")

date_cols_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "review_creation_date",
    "review_answer_timestamp"
]

for col in date_cols_orders:
    if col in fact_orders.columns:
        fact_orders[col] = pd.to_datetime(fact_orders[col], errors="coerce")

In [4]:
print("fact_order_items shape:", fact_order_items.shape)
print("fact_orders shape:", fact_orders.shape)

fact_order_items shape: (110197, 29)
fact_orders shape: (96478, 28)


In [5]:
fact_order_items.head(3)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,item_revenue,total_revenue,customer_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_city,customer_state,product_category_name,product_category_name_english,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,order_month,order_year
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29,58.9,72.19,3ce436f183e68e07877b285a838db11a,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,871766c5855e863f6eccc05f988b23cb,campos dos goytacazes,RJ,cool_stuff,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,2017-09,2017
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93,239.9,259.83,f6dd3ec061db4e3987629fe6b26e5cce,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,eb28e67c4c0b83846050ddfb8a35d051,santa fe do sul,SP,pet_shop,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,2017-04,2017
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87,199.0,216.87,6489ae5e4333f3693df5ad4372dab6d3,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,3818d81c6709e39d06b2738a8d3a2474,para de minas,MG,moveis_decoracao,furniture_decor,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,2018-01,2018


In [6]:
fact_orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_city,customer_state,order_revenue,n_items,n_sellers,n_categories,order_month,order_year,delivery_lead_time_days,estimated_vs_actual_days,is_late,review_score,n_reviews,review_creation_date,review_answer_timestamp,has_review,is_low_review,is_neutral_review,is_high_review
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,38.71,1,1,1,2017-10,2017,8.0,-8.0,0,4.0,1,2017-10-11,2017-10-12 03:43:48,1,0,0,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,barreiras,BA,141.46,1,1,1,2018-07,2018,13.0,-6.0,0,4.0,1,2018-08-08,2018-08-08 18:37:50,1,0,0,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,179.12,1,1,1,2018-08,2018,9.0,-18.0,0,5.0,1,2018-08-18,2018-08-22 19:07:58,1,0,0,1


In [7]:
assert fact_orders["order_id"].nunique() == len(fact_orders), "fact_orders is not one row per order"

## 3. Core KPI Foundation

The first step of the analysis is to establish a compact set of global KPIs describing:

- the scale of the business
- the size of the customer base
- the average commercial value per order
- the overall level of delivery friction
- the overall level of observed customer feedback

To preserve methodological consistency:

- revenue-related KPIs are computed from **fact_order_items**
- delivery and review-related KPIs are computed from **fact_orders**

### 3.1 Commercial scale KPIs

In [8]:
total_revenue = fact_order_items["total_revenue"].sum()
total_orders = fact_orders["order_id"].nunique()
total_customers = fact_orders["customer_unique_id"].nunique()
total_items = len(fact_order_items)
average_order_value = fact_orders["order_revenue"].mean()

commercial_kpis = pd.DataFrame({
    "metric": [
        "Total revenue",
        "Total orders",
        "Total customers",
        "Total items sold",
        "Average order value"
    ],
    "value": [
        round(total_revenue, 2),
        total_orders,
        total_customers,
        total_items,
        round(average_order_value, 2)
    ]
})

commercial_kpis

,metric,value
0,Total revenue,15419773.75
1,Total orders,96478.00
2,Total customers,93358.00
3,Total items sold,110197.00
4,Average order value,159.83


### 3.2 Delivery and feedback KPIs

In [9]:
late_rate = fact_orders["is_late"].mean() * 100
avg_delivery_lead_time = fact_orders["delivery_lead_time_days"].mean()
avg_estimated_vs_actual = fact_orders["estimated_vs_actual_days"].mean()
review_coverage = fact_orders["has_review"].mean() * 100
avg_review_score = fact_orders["review_score"].mean()
low_review_rate = fact_orders["is_low_review"].mean() * 100
neutral_review_rate = fact_orders["is_neutral_review"].mean() * 100
high_review_rate = fact_orders["is_high_review"].mean() * 100

execution_feedback_kpis = pd.DataFrame({
    "metric": [
        "Late delivery rate (%)",
        "Average delivery lead time (days)",
        "Average estimated vs actual (days)",
        "Review coverage (%)",
        "Average review score",
        "Low review rate (%)",
        "Neutral review rate (%)",
        "High review rate (%)"
    ],
    "value": [
        round(late_rate, 2),
        round(avg_delivery_lead_time, 2),
        round(avg_estimated_vs_actual, 2),
        round(review_coverage, 2),
        round(avg_review_score, 2),
        round(low_review_rate, 2),
        round(neutral_review_rate, 2),
        round(high_review_rate, 2)
    ]
})

execution_feedback_kpis

,metric,value
0,Late delivery rate (%),6.77
1,Average delivery lead time (days),12.09
2,Average estimated vs actual (days),-11.88
3,Review coverage (%),99.33
4,Average review score,4.16
5,Low review rate (%),12.71
6,Neutral review rate (%),8.23
7,High review rate (%),78.39


### 3.3 Combined KPI snapshot

In [10]:
kpi_snapshot = pd.concat([commercial_kpis, execution_feedback_kpis], ignore_index=True)
kpi_snapshot

,metric,value
0,Total revenue,15419773.75
1,Total orders,96478.00
2,Total customers,93358.00
3,Total items sold,110197.00
4,Average order value,159.83
5,Late delivery rate (%),6.77
6,Average delivery lead time (days),12.09
7,Average estimated vs actual (days),-11.88
8,Review coverage (%),99.33
9,Average review score,4.16


### Interpretation of the core KPI foundation

The global KPI baseline suggests that the delivered-order scope captures a business of meaningful commercial scale, with more than **15.4M** in revenue, nearly **96.5k** orders, and more than **93k** customers.

The gap between total orders and total customers already points to a **broad customer base with limited repeat activity**, which will need to be examined more explicitly in the customer structure section.

The average order value of roughly **160** provides a useful commercial benchmark for later comparisons across segments, categories, and customer types.

From an operational standpoint, the overall late delivery rate remains **non-negligible but clearly minority**, indicating that most delivered orders are fulfilled on time relative to the promised delivery date.

The average value of `estimated_vs_actual_days` is strongly negative, which indicates that orders are, on average, delivered **well before the estimated delivery date**. This suggests that estimated delivery promises are globally conservative.

Customer feedback is globally positive within the retained delivered-order scope, as shown by the high review coverage, the average review score above **4**, and the strong dominance of high-review outcomes.

At this stage, however, global averages are only a starting point. The real analytical question is whether revenue, friction, and weaker feedback signals become more concentrated in specific markets, segments, or parts of the portfolio.

## 4. Revenue Structure

The next step is to examine how revenue is distributed across time, geography, and product categories.

At this stage, the objective is not yet to interpret operational risk or customer sentiment, but simply to understand the **commercial structure of the business**.

All revenue-related analyses in this section are computed from **fact_order_items**.

### 4.1 Revenue over time

In [11]:
revenue_by_month = (
    fact_order_items
    .groupby("order_month", as_index=False)
    .agg(
        revenue=("total_revenue", "sum"),
        orders=("order_id", "nunique"),
        customers=("customer_unique_id", "nunique"),
        items_sold=("order_item_id", "count")
    )
    .sort_values("order_month")
)

revenue_by_month["average_order_value"] = revenue_by_month["revenue"] / revenue_by_month["orders"]

revenue_by_month.head(10)

,order_month,revenue,orders,customers,items_sold,average_order_value
0,2016-09,143.46,1,1,3,143.460000
1,2016-10,46490.66,265,262,313,175.436453
2,2016-12,19.62,1,1,1,19.620000
3,2017-01,127482.37,750,718,913,169.976493
4,2017-02,271239.32,1653,1630,1858,164.089123
5,2017-03,414330.95,2546,2508,2897,162.738001
6,2017-04,390812.40,2303,2274,2569,169.697091
7,2017-05,566851.40,3546,3479,4004,159.856571
8,2017-06,490050.37,3135,3076,3489,156.315907
9,2017-07,566299.08,3872,3802,4416,146.254928


In [12]:
revenue_by_month.tail(10)

,order_month,revenue,orders,customers,items_sold,average_order_value
13,2017-11,1153364.20,7289,7183,8475,158.233530
14,2017-12,843078.29,5513,5450,6187,152.925502
15,2018-01,1077887.46,7069,6974,8037,152.480897
16,2018-02,966168.41,6555,6400,7518,147.394113
17,2018-03,1120598.24,7003,6914,8017,160.016884
18,2018-04,1132878.93,6798,6744,7827,166.648857
19,2018-05,1128774.52,6749,6693,7810,167.250633
20,2018-06,1011978.29,6099,6061,7010,165.925281
21,2018-07,1027807.28,6159,6100,6963,166.878922
22,2018-08,985491.64,6351,6310,7142,155.171097


### 4.2 Revenue by state

In [13]:
revenue_by_state = (
    fact_order_items
    .groupby("customer_state", as_index=False)
    .agg(
        revenue=("total_revenue", "sum"),
        orders=("order_id", "nunique"),
        customers=("customer_unique_id", "nunique")
    )
    .sort_values("revenue", ascending=False)
)

revenue_by_state["revenue_share_pct"] = (
    revenue_by_state["revenue"] / revenue_by_state["revenue"].sum()
) * 100

revenue_by_state.head(10)

,customer_state,revenue,orders,customers,revenue_share_pct
25,SP,5769703.15,40501,39156,37.417560
18,RJ,2055401.57,12350,11917,13.329648
10,MG,1818891.67,11354,11001,11.795839
22,RS,861472.79,5345,5168,5.586806
17,PR,781708.80,4923,4769,5.069522
23,SC,595127.78,3546,3449,3.859510
4,BA,591137.81,3256,3158,3.833635
6,DF,346123.35,2080,2019,2.244672
8,GO,334212.35,1957,1895,2.167427
7,ES,317657.93,1995,1928,2.060069


### 4.3 Revenue by category

In [14]:
revenue_by_category = (
    fact_order_items
    .groupby("product_category_name_english", as_index=False)
    .agg(
        revenue=("total_revenue", "sum"),
        orders=("order_id", "nunique"),
        items_sold=("order_item_id", "count")
    )
    .sort_values("revenue", ascending=False)
)

revenue_by_category["revenue_share_pct"] = (
    revenue_by_category["revenue"] / revenue_by_category["revenue"].sum()
) * 100

revenue_by_category.head(10)

,product_category_name_english,revenue,orders,items_sold,revenue_share_pct
43,health_beauty,1412089.53,8647,9465,9.280038
70,watches_gifts,1264333.12,5495,5859,8.309005
7,bed_bath_table,1225209.26,9272,10953,8.051889
65,sports_leisure,1118256.91,7530,8431,7.349015
15,computers_accessories,1032723.77,6530,7644,6.786904
39,furniture_decor,880329.92,6307,8160,5.785394
49,housewares,758392.25,5743,6795,4.984039
20,cool_stuff,691680.89,3559,3718,4.545622
5,auto,669454.75,3810,4140,4.399555
42,garden_tools,567145.68,3448,4268,3.727195


### 4.4 Revenue concentration checks

In [15]:
top_state_concentration = pd.DataFrame({
    "metric": [
        "Top 1 state revenue share (%)",
        "Top 3 states revenue share (%)",
        "Top 5 states revenue share (%)",
        "Top 10 states revenue share (%)"
    ],
    "value": [
        round(revenue_by_state["revenue_share_pct"].head(1).sum(), 2),
        round(revenue_by_state["revenue_share_pct"].head(3).sum(), 2),
        round(revenue_by_state["revenue_share_pct"].head(5).sum(), 2),
        round(revenue_by_state["revenue_share_pct"].head(10).sum(), 2)
    ]
})

top_state_concentration

,metric,value
0,Top 1 state revenue share (%),37.42
1,Top 3 states revenue share (%),62.54
2,Top 5 states revenue share (%),73.20
3,Top 10 states revenue share (%),87.36


In [16]:
top_category_concentration = pd.DataFrame({
    "metric": [
        "Top 1 category revenue share (%)",
        "Top 5 categories revenue share (%)",
        "Top 10 categories revenue share (%)",
        "Top 15 categories revenue share (%)"
    ],
    "value": [
        round(revenue_by_category["revenue_share_pct"].head(1).sum(), 2),
        round(revenue_by_category["revenue_share_pct"].head(5).sum(), 2),
        round(revenue_by_category["revenue_share_pct"].head(10).sum(), 2),
        round(revenue_by_category["revenue_share_pct"].head(15).sum(), 2)
    ]
})

top_category_concentration

,metric,value
0,Top 1 category revenue share (%),9.28
1,Top 5 categories revenue share (%),39.78
2,Top 10 categories revenue share (%),63.22
3,Top 15 categories revenue share (%),77.49


### Interpretation of the revenue structure

The monthly revenue profile shows a strong expansion from early 2017 onward, with a clear acceleration through 2017 and early 2018 before stabilizing at a high level. This suggests that the business reached a materially larger commercial scale over time rather than remaining structurally flat.

Geographically, the revenue base is clearly concentrated. **São Paulo alone accounts for roughly 37.4% of total revenue**, and the top 3 states account for more than **62.5%**. This indicates that commercial exposure is driven heavily by a relatively small set of destination markets.

At the same time, the category structure is concentrated but less extreme than the geographic structure. The largest category represents about **9.3%** of total revenue, while the top 10 categories account for roughly **63.2%**. This suggests that the business is commercially diversified across product categories, but still meaningfully concentrated in a limited number of high-contribution segments.

Overall, the revenue structure appears to be **more geographically concentrated than category-concentrated**. This is an important result, because it implies that market-level exposure is likely to matter more than any single product category when identifying commercially material risk areas.

## 5. Customer Breadth

The next step is to examine the structure of the customer base.

The objective is to assess:
- whether the business relies on a narrow set of heavy customers or on a broad customer base
- how frequent repeat ordering is
- how concentrated revenue is at the customer level

Customer-related analyses in this section combine:
- **fact_orders** for order recurrence
- **fact_order_items** for revenue concentration by customer

### 5.1 Orders per customer

In [17]:
orders_per_customer = (
    fact_orders
    .groupby("customer_unique_id", as_index=False)
    .agg(
        orders=("order_id", "nunique")
    )
    .sort_values("orders", ascending=False)
)

orders_per_customer["customer_type"] = np.where(
    orders_per_customer["orders"] == 1,
    "Single-order",
    "Repeat"
)

orders_per_customer.head(10)

,customer_unique_id,orders,customer_type
51431,8d50f5eadf50201ccdcedfb9e2ac8455,15,Repeat
22779,3e43e6105506432c953e165fb2acf44c,9,Repeat
36706,6469f99c1f9dfae7733b25662e7f1782,7,Repeat
10060,1b6c7548a2a1f9037c1fd3ddfed95f33,7,Repeat
73921,ca77025e7201e3b30c44b472ff346268,7,Repeat
26257,47c1a3033b8b77b3ab6e109eb4d5fdf3,6,Repeat
80538,dc813062e0fc23409cd255f7f53c7074,6,Repeat
36500,63cfc61cee11cbe306bff5857d00bfe4,6,Repeat
87885,f0e310a6839dce9de1638e0fe5ab282a,6,Repeat
6964,12f5d6e1cbf93dafd9dcc19095df0b3d,6,Repeat


In [18]:
orders_per_customer_distribution = (
    orders_per_customer["orders"]
    .value_counts()
    .sort_index()
    .reset_index()
)

orders_per_customer_distribution.columns = ["orders_per_customer", "number_of_customers"]
orders_per_customer_distribution.head(10)

,orders_per_customer,number_of_customers
0,1,90557
1,2,2573
2,3,181
3,4,28
4,5,9
5,6,5
6,7,3
7,9,1
8,15,1


### 5.2 Single-order vs repeat customers

In [19]:
customer_type_summary = (
    orders_per_customer
    .groupby("customer_type", as_index=False)
    .agg(
        customers=("customer_unique_id", "nunique")
    )
)

customer_type_summary["customer_share_pct"] = (
    customer_type_summary["customers"] / customer_type_summary["customers"].sum()
) * 100

customer_type_summary

,customer_type,customers,customer_share_pct
0,Repeat,2801,3.000278
1,Single-order,90557,96.999722


### 5.3 Revenue by customer

In [20]:
revenue_by_customer = (
    fact_order_items
    .groupby("customer_unique_id", as_index=False)
    .agg(
        revenue=("total_revenue", "sum"),
        orders=("order_id", "nunique")
    )
    .sort_values("revenue", ascending=False)
)

revenue_by_customer.head(10)

,customer_unique_id,revenue,orders
3724,0a0a92112bd4c708ca5fde585afaa872,13664.08,1
79636,da122df9eeddfedc1dc1f5349a1a690c,7571.63,2
43168,763c8b1c9c68a0229c42c9fc6f662b93,7274.88,1
80463,dc4802a71eae9be1dd28f5d788ceb526,6929.31,1
25436,459bef486812aa25204be022145caa62,6922.21,1
93081,ff4159b92c40ebe40454e3e6a7c35ed6,6726.66,1
23411,4007669dec559734d6f53e029e360987,6081.54,1
87148,eebb5dda148d3893cdaf5b5ca3040ccb,4764.34,1
26640,48e1ac109decbb87765a3eade6854098,4681.78,1
73127,c8460e4251689ba205045f3ea17884a1,4655.88,4


In [21]:
revenue_by_customer["customer_rank"] = range(1, len(revenue_by_customer) + 1)
revenue_by_customer["customer_percentile"] = (
    revenue_by_customer["customer_rank"] / len(revenue_by_customer)
) * 100

In [22]:
top_customer_concentration = pd.DataFrame({
    "metric": [
        "Top 1% customers revenue share (%)",
        "Top 10% customers revenue share (%)",
        "Remaining 90% customers revenue share (%)"
    ],
    "value": [
        round(
            revenue_by_customer.loc[revenue_by_customer["customer_percentile"] <= 1, "revenue"].sum()
            / revenue_by_customer["revenue"].sum() * 100,
            2
        ),
        round(
            revenue_by_customer.loc[revenue_by_customer["customer_percentile"] <= 10, "revenue"].sum()
            / revenue_by_customer["revenue"].sum() * 100,
            2
        ),
        round(
            revenue_by_customer.loc[revenue_by_customer["customer_percentile"] > 10, "revenue"].sum()
            / revenue_by_customer["revenue"].sum() * 100,
            2
        )
    ]
})

top_customer_concentration

,metric,value
0,Top 1% customers revenue share (%),10.34
1,Top 10% customers revenue share (%),38.25
2,Remaining 90% customers revenue share (%),61.75


### 5.4 Revenue by customer type

In [23]:
customer_type_revenue = (
    revenue_by_customer
    .merge(
        orders_per_customer[["customer_unique_id", "customer_type"]],
        on="customer_unique_id",
        how="left"
    )
    .groupby("customer_type", as_index=False)
    .agg(
        customers=("customer_unique_id", "nunique"),
        revenue=("revenue", "sum"),
        average_revenue_per_customer=("revenue", "mean")
    )
)

customer_type_revenue["revenue_share_pct"] = (
    customer_type_revenue["revenue"] / customer_type_revenue["revenue"].sum()
) * 100

customer_type_revenue

,customer_type,customers,revenue,average_revenue_per_customer,revenue_share_pct
0,Repeat,2801,864187.46,308.528190,5.604411
1,Single-order,90557,14555586.29,160.733972,94.395589


### 5.5 Average order value by customer type

In [24]:
orders_with_customer_type = (
    fact_orders[["order_id", "customer_unique_id", "order_revenue"]]
    .merge(
        orders_per_customer[["customer_unique_id", "customer_type"]],
        on="customer_unique_id",
        how="left"
    )
)

aov_by_customer_type = (
    orders_with_customer_type
    .groupby("customer_type", as_index=False)
    .agg(
        average_order_value=("order_revenue", "mean"),
        orders=("order_id", "nunique")
    )
)

aov_by_customer_type

,customer_type,average_order_value,orders
0,Repeat,145.952957,5921
1,Single-order,160.733972,90557


### Interpretation of the customer breadth

The customer structure is extremely broad and only weakly recurrent. Nearly **97%** of customers place a single order, while only about **3%** return for at least one additional purchase.

This is confirmed by the order distribution itself: the overwhelming majority of customers place exactly **one** order, while repeat behaviors beyond two or three orders remain rare and highly marginal.

Revenue concentration at the customer level exists, but it is limited relative to the geographic concentration observed earlier. The top **1%** of customers account for roughly **10.3%** of total revenue, and the top **10%** account for about **38.3%**. This means that most revenue still comes from the broad customer base rather than from a narrow set of dominant buyers.

Single-order customers generate the vast majority of revenue, accounting for roughly **94.4%** of total revenue, while repeat customers contribute only around **5.6%**. This reinforces the idea that the commercial model is driven much more by breadth of acquisition than by strong repeat monetization.

Interestingly, repeat customers generate more revenue per customer on average, but their average order value is slightly lower than that of single-order customers. This suggests that repeat contribution comes mainly from **frequency**, not from higher-value baskets.

Overall, the business appears to rely on a **very broad and weakly recurrent customer base**, which means that market exposure and operational execution in commercially important regions may matter more than dependence on a narrow set of heavy customers.

## 6. Delivery Friction

The previous sections showed that the revenue base is more strongly concentrated across **destination markets** than across individual customers, and that the business relies on a very broad customer base with limited repeat intensity.

As a result, the first operational question is not whether friction exists in general, but whether it appears in the **markets that matter most commercially**.

This section therefore begins with a **geographic view of delivery friction**, because geography is the most commercially material concentration layer identified so far.

The focus is on:
- lateness relative to the estimated delivery promise
- delivery lead time
- geographic variation in delivery performance

All indicators in this section are computed from **fact_orders**, since delivery outcomes are naturally defined at the order level.

### 6.1 Global delivery profile

In [25]:
delivery_profile = pd.DataFrame({
    "metric": [
        "Late delivery rate (%)",
        "Average delivery lead time (days)",
        "Average estimated vs actual (days)",
        "Median delivery lead time (days)",
        "Median estimated vs actual (days)"
    ],
    "value": [
        round(fact_orders["is_late"].mean() * 100, 2),
        round(fact_orders["delivery_lead_time_days"].mean(), 2),
        round(fact_orders["estimated_vs_actual_days"].mean(), 2),
        round(fact_orders["delivery_lead_time_days"].median(), 2),
        round(fact_orders["estimated_vs_actual_days"].median(), 2)
    ]
})

delivery_profile

,metric,value
0,Late delivery rate (%),6.77
1,Average delivery lead time (days),12.09
2,Average estimated vs actual (days),-11.88
3,Median delivery lead time (days),10.00
4,Median estimated vs actual (days),-12.00


### 6.2 Lateness by state

In [26]:
state_delivery = (
    fact_orders
    .groupby("customer_state", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        late_rate=("is_late", "mean"),
        avg_delivery_lead_time=("delivery_lead_time_days", "mean"),
        avg_estimated_vs_actual=("estimated_vs_actual_days", "mean")
    )
)

state_delivery["late_rate"] = state_delivery["late_rate"] * 100
state_delivery = state_delivery.sort_values("late_rate", ascending=False)

state_delivery.head(10)

,customer_state,orders,late_rate,avg_delivery_lead_time,avg_estimated_vs_actual
1,AL,397,21.410579,24.040302,-8.707809
9,MA,717,17.433752,21.117155,-9.571827
24,SE,335,15.223881,21.029851,-10.020896
16,PI,476,13.865546,18.993697,-11.306723
5,CE,1279,13.760751,20.817826,-10.804535
21,RR,41,12.195122,28.975610,-17.292683
4,BA,3256,12.162162,18.866400,-10.794533
18,RJ,12350,12.105263,14.848583,-11.761215
13,PA,946,11.205074,23.316068,-14.066596
7,ES,1995,10.726817,15.331830,-10.496241


### 6.3 Focus on top revenue states

In [27]:
top_revenue_states = revenue_by_state.head(10)[["customer_state", "revenue", "revenue_share_pct"]].copy()

top_revenue_state_delivery = top_revenue_states.merge(
    state_delivery,
    on="customer_state",
    how="left"
)

top_revenue_state_delivery = top_revenue_state_delivery.sort_values("revenue", ascending=False)
top_revenue_state_delivery

,customer_state,revenue,revenue_share_pct,orders,late_rate,avg_delivery_lead_time,avg_estimated_vs_actual
0,SP,5769703.15,37.417560,40501,4.493716,8.298094,-11.075542
1,RJ,2055401.57,13.329648,12350,12.105263,14.848583,-11.761215
2,MG,1818891.67,11.795839,11354,4.571076,11.542188,-13.242998
3,RS,861472.79,5.586806,5345,6.080449,14.819237,-13.910367
4,PR,781708.80,5.069522,4923,4.042251,11.526711,-13.314239
5,SC,595127.78,3.859510,3546,8.206430,14.475183,-11.503384
6,BA,591137.81,3.833635,3256,12.162162,18.866400,-10.794533
7,DF,346123.35,2.244672,2080,5.673077,12.509135,-12.048077
8,GO,334212.35,2.167427,1957,6.540623,15.150741,-12.185488
9,ES,317657.93,2.060069,1995,10.726817,15.331830,-10.496241


### 6.4 Priority-state friction view

In [28]:
priority_state_friction = top_revenue_state_delivery[[
    "customer_state",
    "revenue",
    "revenue_share_pct",
    "orders",
    "late_rate",
    "avg_delivery_lead_time",
    "avg_estimated_vs_actual"
]].copy()

priority_state_friction

,customer_state,revenue,revenue_share_pct,orders,late_rate,avg_delivery_lead_time,avg_estimated_vs_actual
0,SP,5769703.15,37.417560,40501,4.493716,8.298094,-11.075542
1,RJ,2055401.57,13.329648,12350,12.105263,14.848583,-11.761215
2,MG,1818891.67,11.795839,11354,4.571076,11.542188,-13.242998
3,RS,861472.79,5.586806,5345,6.080449,14.819237,-13.910367
4,PR,781708.80,5.069522,4923,4.042251,11.526711,-13.314239
5,SC,595127.78,3.859510,3546,8.206430,14.475183,-11.503384
6,BA,591137.81,3.833635,3256,12.162162,18.866400,-10.794533
7,DF,346123.35,2.244672,2080,5.673077,12.509135,-12.048077
8,GO,334212.35,2.167427,1957,6.540623,15.150741,-12.185488
9,ES,317657.93,2.060069,1995,10.726817,15.331830,-10.496241


### 6.5 Friction benchmark against the global average

In [29]:
global_late_rate = fact_orders["is_late"].mean() * 100

priority_state_friction["late_rate_vs_global"] = (
    priority_state_friction["late_rate"] - global_late_rate
)

priority_state_friction = priority_state_friction.sort_values(
    ["late_rate_vs_global", "revenue"],
    ascending=[False, False]
)

priority_state_friction

,customer_state,revenue,revenue_share_pct,orders,late_rate,avg_delivery_lead_time,avg_estimated_vs_actual,late_rate_vs_global
6,BA,591137.81,3.833635,3256,12.162162,18.866400,-10.794533,5.389634
1,RJ,2055401.57,13.329648,12350,12.105263,14.848583,-11.761215,5.332735
9,ES,317657.93,2.060069,1995,10.726817,15.331830,-10.496241,3.954289
5,SC,595127.78,3.859510,3546,8.206430,14.475183,-11.503384,1.433901
8,GO,334212.35,2.167427,1957,6.540623,15.150741,-12.185488,-0.231905
3,RS,861472.79,5.586806,5345,6.080449,14.819237,-13.910367,-0.692079
7,DF,346123.35,2.244672,2080,5.673077,12.509135,-12.048077,-1.099452
2,MG,1818891.67,11.795839,11354,4.571076,11.542188,-13.242998,-2.201452
0,SP,5769703.15,37.417560,40501,4.493716,8.298094,-11.075542,-2.278812
4,PR,781708.80,5.069522,4923,4.042251,11.526711,-13.314239,-2.730278


### Interpretation of the delivery friction

At the global level, operational execution appears broadly stable: only about **6.8%** of delivered orders are late, and orders are delivered on average roughly **12 days before** the estimated delivery date. This confirms that lateness is not a mass issue across the full delivered-order population.

However, the geographic breakdown shows that friction is not evenly distributed. Some states exhibit materially higher late-delivery rates than the global average, while others remain clearly below it.

Among the top revenue states, the most important result is the overlap between **commercial weight** and **elevated lateness**. In particular, **RJ** stands out as a major market with a late-delivery rate well above the global benchmark. **BA** and **ES** also show materially higher lateness while still representing non-negligible commercial exposure. **SC** appears above the global average as well, though with a smaller revenue base than RJ.

By contrast, **SP**, despite dominating the revenue base, performs better than the global lateness average. This makes it a commercially important but operationally healthier anchor. **MG** and **PR** also appear relatively healthier on this dimension.

Overall, the friction view suggests that operational risk should not be framed as a broad systemic problem, but rather as a **selective exposure issue concentrated in specific commercially relevant markets**. This supports a prioritization logic based on the overlap between revenue concentration and weaker execution quality.

## 7. Customer Feedback

The next step is to examine review-based customer feedback within the delivered-order scope.

The purpose of this section is to assess:
- the overall profile of observed reviews
- whether late orders are associated with weaker feedback outcomes
- whether review-based signals can complement the delivery-friction analysis

All indicators in this section are computed from **fact_orders**, since reviews are naturally defined at the order level.

### 7.1 Global feedback profile

In [30]:
feedback_profile = pd.DataFrame({
    "metric": [
        "Review coverage (%)",
        "Average review score",
        "Low review rate (%)",
        "Neutral review rate (%)",
        "High review rate (%)"
    ],
    "value": [
        round(fact_orders["has_review"].mean() * 100, 2),
        round(fact_orders["review_score"].mean(), 2),
        round(fact_orders["is_low_review"].mean() * 100, 2),
        round(fact_orders["is_neutral_review"].mean() * 100, 2),
        round(fact_orders["is_high_review"].mean() * 100, 2)
    ]
})

feedback_profile

,metric,value
0,Review coverage (%),99.33
1,Average review score,4.16
2,Low review rate (%),12.71
3,Neutral review rate (%),8.23
4,High review rate (%),78.39


### 7.2 Feedback by lateness status

In [31]:
feedback_by_lateness = (
    fact_orders
    .groupby("is_late", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        avg_review_score=("review_score", "mean"),
        low_review_rate=("is_low_review", "mean"),
        neutral_review_rate=("is_neutral_review", "mean"),
        high_review_rate=("is_high_review", "mean"),
        review_coverage=("has_review", "mean")
    )
)

feedback_by_lateness["low_review_rate"] *= 100
feedback_by_lateness["neutral_review_rate"] *= 100
feedback_by_lateness["high_review_rate"] *= 100
feedback_by_lateness["review_coverage"] *= 100

feedback_by_lateness

,is_late,orders,avg_review_score,low_review_rate,neutral_review_rate,high_review_rate,review_coverage
0,0,89944,4.290608,9.209063,8.058347,82.184470,99.451881
1,1,6534,2.271823,60.973370,10.575451,26.109581,97.658402


### 7.3 Feedback gap between late and on-time orders

In [32]:
on_time_feedback = feedback_by_lateness.loc[feedback_by_lateness["is_late"] == 0].iloc[0]
late_feedback = feedback_by_lateness.loc[feedback_by_lateness["is_late"] == 1].iloc[0]

feedback_gap = pd.DataFrame({
    "metric": [
        "Review score gap (late - on-time)",
        "Low review rate gap (late - on-time, pp)",
        "Neutral review rate gap (late - on-time, pp)",
        "High review rate gap (late - on-time, pp)"
    ],
    "value": [
        round(late_feedback["avg_review_score"] - on_time_feedback["avg_review_score"], 2),
        round(late_feedback["low_review_rate"] - on_time_feedback["low_review_rate"], 2),
        round(late_feedback["neutral_review_rate"] - on_time_feedback["neutral_review_rate"], 2),
        round(late_feedback["high_review_rate"] - on_time_feedback["high_review_rate"], 2)
    ]
})

feedback_gap

,metric,value
0,Review score gap (late - on-time),-2.02
1,"Low review rate gap (late - on-time, pp)",51.76
2,"Neutral review rate gap (late - on-time, pp)",2.52
3,"High review rate gap (late - on-time, pp)",-56.07


### 7.4 Feedback by state

In [33]:
state_feedback = (
    fact_orders
    .groupby("customer_state", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        review_coverage=("has_review", "mean"),
        avg_review_score=("review_score", "mean"),
        low_review_rate=("is_low_review", "mean"),
        neutral_review_rate=("is_neutral_review", "mean"),
        high_review_rate=("is_high_review", "mean")
    )
)

state_feedback["review_coverage"] *= 100
state_feedback["low_review_rate"] *= 100
state_feedback["neutral_review_rate"] *= 100
state_feedback["high_review_rate"] *= 100

state_feedback.sort_values("avg_review_score", ascending=True).head(10)

,customer_state,orders,review_coverage,avg_review_score,low_review_rate,neutral_review_rate,high_review_rate
9,MA,717,99.302650,3.833567,19.804742,9.762901,69.735007
1,AL,397,99.244332,3.852792,20.906801,7.304786,71.032746
21,RR,41,100.000000,3.902439,14.634146,17.073171,68.292683
24,SE,335,99.701493,3.907186,18.805970,8.656716,72.238806
13,PA,946,98.625793,3.909968,17.653277,9.936575,71.035941
4,BA,3256,99.170762,3.930009,16.891892,10.012285,72.266585
5,CE,1279,99.530884,3.941870,17.200938,10.007819,72.322127
18,RJ,12350,98.874494,3.965141,18.129555,8.178138,72.566802
16,PI,476,98.949580,3.993631,15.966387,8.613445,74.369748
14,PB,517,99.032882,4.076172,14.700193,9.090909,75.241779


### 7.5 Feedback profile for top revenue states

In [34]:
top_revenue_state_feedback = top_revenue_states.merge(
    state_feedback,
    on="customer_state",
    how="left"
)

top_revenue_state_feedback = top_revenue_state_feedback.sort_values("revenue", ascending=False)
top_revenue_state_feedback

,customer_state,revenue,revenue_share_pct,orders,review_coverage,avg_review_score,low_review_rate,neutral_review_rate,high_review_rate
0,SP,5769703.15,37.417560,40501,99.437051,4.246551,10.609615,7.886225,80.941211
1,RJ,2055401.57,13.329648,12350,98.874494,3.965141,18.129555,8.178138,72.566802
2,MG,1818891.67,11.795839,11354,99.392285,4.192468,11.652281,8.358288,79.381716
3,RS,861472.79,5.586806,5345,99.663237,4.185752,11.880262,8.213283,79.569691
4,PR,781708.80,5.069522,4923,99.532805,4.239592,10.806419,7.576681,81.149705
5,SC,595127.78,3.859510,3546,99.238579,4.133276,12.915962,8.798646,77.523971
6,BA,591137.81,3.833635,3256,99.170762,3.930009,16.891892,10.012285,72.266585
7,DF,346123.35,2.244672,2080,99.519231,4.133816,13.173077,7.884615,78.461538
8,GO,334212.35,2.167427,1957,99.437915,4.104317,13.132345,9.606541,76.699029
9,ES,317657.93,2.060069,1995,98.696742,4.079736,13.634085,8.972431,76.090226


### Interpretation of the customer feedback

The global feedback profile is strongly positive: review coverage is nearly complete, the average review score is above **4.1**, and high-review outcomes clearly dominate the delivered-order population.

However, the comparison between late and on-time orders shows a very strong observational gap. Late orders have an average review score lower by about **2 points**, a dramatically higher low-review rate, and a much lower high-review rate than on-time orders.

This does not prove causality on its own, but it clearly indicates that delivery friction is not only an operational signal: it is also associated with materially weaker observed customer feedback.

At the state level, the top revenue markets do not all behave the same way. **SP** combines very high commercial weight with relatively strong feedback outcomes, which reinforces its role as a healthy commercial anchor. By contrast, **RJ** and **BA** show weaker average review scores and higher low-review rates, which makes their operational fragility more commercially relevant. **ES** also appears weaker than the healthier core states, while **MG**, **RS**, and **PR** look comparatively more stable from a feedback standpoint.

Overall, the review layer strengthens the project considerably: markets with higher delivery friction are not just operationally weaker, but often also associated with a weaker post-purchase feedback signal. This makes the notion of **commercially material execution risk** much more defensible.

## 8. Combined State View: Revenue, Friction, and Feedback

The next step is to consolidate the three main analytical dimensions identified so far at the state level:

- **commercial weight** through revenue exposure
- **operational execution** through delivery friction
- **customer signal** through review-based feedback

This combined state view is designed to identify where operational and feedback weaknesses overlap with commercially material exposure.

### 8.1 Build the combined state table

In [42]:
state_combined = (
    revenue_by_state
    .merge(
        state_delivery[[
            "customer_state",
            "late_rate",
            "avg_delivery_lead_time",
            "avg_estimated_vs_actual"
        ]],
        on="customer_state",
        how="left"
    )
    .merge(
        state_feedback[[
            "customer_state",
            "review_coverage",
            "avg_review_score",
            "low_review_rate",
            "neutral_review_rate",
            "high_review_rate"
        ]],
        on="customer_state",
        how="left"
    )
)

state_combined = state_combined.sort_values("revenue", ascending=False)
state_combined.head(10)

,customer_state,revenue,orders,customers,revenue_share_pct,late_rate,avg_delivery_lead_time,avg_estimated_vs_actual,review_coverage,avg_review_score,low_review_rate,neutral_review_rate,high_review_rate
0,SP,5769703.15,40501,39156,37.417560,4.493716,8.298094,-11.075542,99.437051,4.246551,10.609615,7.886225,80.941211
1,RJ,2055401.57,12350,11917,13.329648,12.105263,14.848583,-11.761215,98.874494,3.965141,18.129555,8.178138,72.566802
2,MG,1818891.67,11354,11001,11.795839,4.571076,11.542188,-13.242998,99.392285,4.192468,11.652281,8.358288,79.381716
3,RS,861472.79,5345,5168,5.586806,6.080449,14.819237,-13.910367,99.663237,4.185752,11.880262,8.213283,79.569691
4,PR,781708.80,4923,4769,5.069522,4.042251,11.526711,-13.314239,99.532805,4.239592,10.806419,7.576681,81.149705
5,SC,595127.78,3546,3449,3.859510,8.206430,14.475183,-11.503384,99.238579,4.133276,12.915962,8.798646,77.523971
6,BA,591137.81,3256,3158,3.833635,12.162162,18.866400,-10.794533,99.170762,3.930009,16.891892,10.012285,72.266585
7,DF,346123.35,2080,2019,2.244672,5.673077,12.509135,-12.048077,99.519231,4.133816,13.173077,7.884615,78.461538
8,GO,334212.35,1957,1895,2.167427,6.540623,15.150741,-12.185488,99.437915,4.104317,13.132345,9.606541,76.699029
9,ES,317657.93,1995,1928,2.060069,10.726817,15.331830,-10.496241,98.696742,4.079736,13.634085,8.972431,76.090226


### 8.2 Benchmark against global averages

In [43]:
global_late_rate = fact_orders["is_late"].mean() * 100
global_avg_review_score = fact_orders["review_score"].mean()
global_low_review_rate = fact_orders["is_low_review"].mean() * 100

state_combined["late_rate_vs_global"] = state_combined["late_rate"] - global_late_rate
state_combined["review_score_vs_global"] = state_combined["avg_review_score"] - global_avg_review_score
state_combined["low_review_rate_vs_global"] = state_combined["low_review_rate"] - global_low_review_rate

state_combined.head(10)

,customer_state,revenue,orders,customers,revenue_share_pct,late_rate,avg_delivery_lead_time,avg_estimated_vs_actual,review_coverage,avg_review_score,low_review_rate,neutral_review_rate,high_review_rate,late_rate_vs_global,review_score_vs_global,low_review_rate_vs_global
0,SP,5769703.15,40501,39156,37.417560,4.493716,8.298094,-11.075542,99.437051,4.246551,10.609615,7.886225,80.941211,-2.278812,0.090364,-2.105201
1,RJ,2055401.57,12350,11917,13.329648,12.105263,14.848583,-11.761215,98.874494,3.965141,18.129555,8.178138,72.566802,5.332735,-0.191046,5.414739
2,MG,1818891.67,11354,11001,11.795839,4.571076,11.542188,-13.242998,99.392285,4.192468,11.652281,8.358288,79.381716,-2.201452,0.036281,-1.062535
3,RS,861472.79,5345,5168,5.586806,6.080449,14.819237,-13.910367,99.663237,4.185752,11.880262,8.213283,79.569691,-0.692079,0.029565,-0.834554
4,PR,781708.80,4923,4769,5.069522,4.042251,11.526711,-13.314239,99.532805,4.239592,10.806419,7.576681,81.149705,-2.730278,0.083405,-1.908397
5,SC,595127.78,3546,3449,3.859510,8.206430,14.475183,-11.503384,99.238579,4.133276,12.915962,8.798646,77.523971,1.433901,-0.022910,0.201146
6,BA,591137.81,3256,3158,3.833635,12.162162,18.866400,-10.794533,99.170762,3.930009,16.891892,10.012285,72.266585,5.389634,-0.226177,4.177076
7,DF,346123.35,2080,2019,2.244672,5.673077,12.509135,-12.048077,99.519231,4.133816,13.173077,7.884615,78.461538,-1.099452,-0.022370,0.458261
8,GO,334212.35,1957,1895,2.167427,6.540623,15.150741,-12.185488,99.437915,4.104317,13.132345,9.606541,76.699029,-0.231905,-0.051870,0.417530
9,ES,317657.93,1995,1928,2.060069,10.726817,15.331830,-10.496241,98.696742,4.079736,13.634085,8.972431,76.090226,3.954289,-0.076451,0.919269


### 8.3 Focus on top revenue states

In [44]:
top_revenue_state_combined = state_combined.head(10).copy()
top_revenue_state_combined

,customer_state,revenue,orders,customers,revenue_share_pct,late_rate,avg_delivery_lead_time,avg_estimated_vs_actual,review_coverage,avg_review_score,low_review_rate,neutral_review_rate,high_review_rate,late_rate_vs_global,review_score_vs_global,low_review_rate_vs_global
0,SP,5769703.15,40501,39156,37.417560,4.493716,8.298094,-11.075542,99.437051,4.246551,10.609615,7.886225,80.941211,-2.278812,0.090364,-2.105201
1,RJ,2055401.57,12350,11917,13.329648,12.105263,14.848583,-11.761215,98.874494,3.965141,18.129555,8.178138,72.566802,5.332735,-0.191046,5.414739
2,MG,1818891.67,11354,11001,11.795839,4.571076,11.542188,-13.242998,99.392285,4.192468,11.652281,8.358288,79.381716,-2.201452,0.036281,-1.062535
3,RS,861472.79,5345,5168,5.586806,6.080449,14.819237,-13.910367,99.663237,4.185752,11.880262,8.213283,79.569691,-0.692079,0.029565,-0.834554
4,PR,781708.80,4923,4769,5.069522,4.042251,11.526711,-13.314239,99.532805,4.239592,10.806419,7.576681,81.149705,-2.730278,0.083405,-1.908397
5,SC,595127.78,3546,3449,3.859510,8.206430,14.475183,-11.503384,99.238579,4.133276,12.915962,8.798646,77.523971,1.433901,-0.022910,0.201146
6,BA,591137.81,3256,3158,3.833635,12.162162,18.866400,-10.794533,99.170762,3.930009,16.891892,10.012285,72.266585,5.389634,-0.226177,4.177076
7,DF,346123.35,2080,2019,2.244672,5.673077,12.509135,-12.048077,99.519231,4.133816,13.173077,7.884615,78.461538,-1.099452,-0.022370,0.458261
8,GO,334212.35,1957,1895,2.167427,6.540623,15.150741,-12.185488,99.437915,4.104317,13.132345,9.606541,76.699029,-0.231905,-0.051870,0.417530
9,ES,317657.93,1995,1928,2.060069,10.726817,15.331830,-10.496241,98.696742,4.079736,13.634085,8.972431,76.090226,3.954289,-0.076451,0.919269


### 8.4 Priority state screening logic

In [45]:
priority_state_screen = top_revenue_state_combined[[
    "customer_state",
    "revenue",
    "revenue_share_pct",
    "late_rate",
    "late_rate_vs_global",
    "avg_review_score",
    "review_score_vs_global",
    "low_review_rate",
    "low_review_rate_vs_global"
]].copy()

priority_state_screen

,customer_state,revenue,revenue_share_pct,late_rate,late_rate_vs_global,avg_review_score,review_score_vs_global,low_review_rate,low_review_rate_vs_global
0,SP,5769703.15,37.417560,4.493716,-2.278812,4.246551,0.090364,10.609615,-2.105201
1,RJ,2055401.57,13.329648,12.105263,5.332735,3.965141,-0.191046,18.129555,5.414739
2,MG,1818891.67,11.795839,4.571076,-2.201452,4.192468,0.036281,11.652281,-1.062535
3,RS,861472.79,5.586806,6.080449,-0.692079,4.185752,0.029565,11.880262,-0.834554
4,PR,781708.80,5.069522,4.042251,-2.730278,4.239592,0.083405,10.806419,-1.908397
5,SC,595127.78,3.859510,8.206430,1.433901,4.133276,-0.022910,12.915962,0.201146
6,BA,591137.81,3.833635,12.162162,5.389634,3.930009,-0.226177,16.891892,4.177076
7,DF,346123.35,2.244672,5.673077,-1.099452,4.133816,-0.022370,13.173077,0.458261
8,GO,334212.35,2.167427,6.540623,-0.231905,4.104317,-0.051870,13.132345,0.417530
9,ES,317657.93,2.060069,10.726817,3.954289,4.079736,-0.076451,13.634085,0.919269


### 8.5 Exposure-risk ordering

In [46]:
priority_state_screen = priority_state_screen.sort_values(
    by=["late_rate_vs_global", "low_review_rate_vs_global", "revenue"],
    ascending=[False, False, False]
)

priority_state_screen

,customer_state,revenue,revenue_share_pct,late_rate,late_rate_vs_global,avg_review_score,review_score_vs_global,low_review_rate,low_review_rate_vs_global
6,BA,591137.81,3.833635,12.162162,5.389634,3.930009,-0.226177,16.891892,4.177076
1,RJ,2055401.57,13.329648,12.105263,5.332735,3.965141,-0.191046,18.129555,5.414739
9,ES,317657.93,2.060069,10.726817,3.954289,4.079736,-0.076451,13.634085,0.919269
5,SC,595127.78,3.859510,8.206430,1.433901,4.133276,-0.022910,12.915962,0.201146
8,GO,334212.35,2.167427,6.540623,-0.231905,4.104317,-0.051870,13.132345,0.417530
3,RS,861472.79,5.586806,6.080449,-0.692079,4.185752,0.029565,11.880262,-0.834554
7,DF,346123.35,2.244672,5.673077,-1.099452,4.133816,-0.022370,13.173077,0.458261
2,MG,1818891.67,11.795839,4.571076,-2.201452,4.192468,0.036281,11.652281,-1.062535
0,SP,5769703.15,37.417560,4.493716,-2.278812,4.246551,0.090364,10.609615,-2.105201
4,PR,781708.80,5.069522,4.042251,-2.730278,4.239592,0.083405,10.806419,-1.908397


### 8.6 Healthy-anchor view

In [47]:
healthy_anchor_states = top_revenue_state_combined[
    (top_revenue_state_combined["late_rate_vs_global"] < 0)
    & (top_revenue_state_combined["review_score_vs_global"] > 0)
].copy()

healthy_anchor_states = healthy_anchor_states.sort_values("revenue", ascending=False)
healthy_anchor_states

,customer_state,revenue,orders,customers,revenue_share_pct,late_rate,avg_delivery_lead_time,avg_estimated_vs_actual,review_coverage,avg_review_score,low_review_rate,neutral_review_rate,high_review_rate,late_rate_vs_global,review_score_vs_global,low_review_rate_vs_global
0,SP,5769703.15,40501,39156,37.417560,4.493716,8.298094,-11.075542,99.437051,4.246551,10.609615,7.886225,80.941211,-2.278812,0.090364,-2.105201
2,MG,1818891.67,11354,11001,11.795839,4.571076,11.542188,-13.242998,99.392285,4.192468,11.652281,8.358288,79.381716,-2.201452,0.036281,-1.062535
3,RS,861472.79,5345,5168,5.586806,6.080449,14.819237,-13.910367,99.663237,4.185752,11.880262,8.213283,79.569691,-0.692079,0.029565,-0.834554
4,PR,781708.80,4923,4769,5.069522,4.042251,11.526711,-13.314239,99.532805,4.239592,10.806419,7.576681,81.149705,-2.730278,0.083405,-1.908397


### Interpretation of the combined state view

The combined state view confirms that commercial exposure, operational friction, and customer feedback do not align uniformly across markets.

A first group of states stands out as **priority fragile markets**, where operational weakness overlaps with commercially meaningful exposure. **RJ** is the clearest case: it combines large revenue weight with a late-delivery rate far above the global average, weaker review scores, and a materially higher low-review rate. **BA** shows a very similar pattern, although with a smaller revenue base. **ES** also appears weaker than the global benchmark on both friction and feedback, while **SC** remains somewhat above average on lateness with slightly weaker feedback.

A second group of states acts more like **healthy commercial anchors**. **SP** is the strongest example: it dominates the revenue base, yet performs better than the global lateness benchmark and also shows stronger review outcomes than average. **MG**, **RS**, and **PR** also appear comparatively healthier, combining meaningful commercial contribution with better execution and/or better review signals.

This state-level synthesis strengthens the project’s main strategic conclusion: the issue is not broad underperformance across the whole market footprint, but rather a **selective overlap between commercial weight and execution fragility** concentrated in a limited number of priority markets.

In practical terms, this means that operational remediation should be prioritized where three conditions meet:
- meaningful revenue exposure
- above-average lateness
- weaker-than-average customer feedback

That logic points primarily to **RJ** and **BA**, with **ES** and **SC** as secondary fragile markets, while **SP**, **MG**, **RS**, and **PR** behave more like operationally healthier core anchors.

## 9. Portfolio Insights

The final analytical angle focuses on the structure of the commercial portfolio.

This section examines how revenue is distributed across:
- product categories
- sellers

The objective is not to assign pure causal responsibility for delivery performance or customer satisfaction to categories or sellers, but to identify where the portfolio is commercially important and more exposed to:
- late-order risk
- weaker review-based feedback outcomes

All revenue-related analyses in this section are computed from **fact_order_items**, while delivery and feedback exposure signals are brought in from **fact_orders** at the order level.

### 9.1 Methodological note

The item-level analytical table does not contain delivery or review KPIs directly, since both delivery outcomes and reviews are naturally defined at the order level.

To assess portfolio exposure, a temporary working table is created by merging the item-level revenue structure with selected order-level signals from **fact_orders**:

- `is_late`
- `review_score`
- `is_low_review`

This preserves a clean distinction between:
- item-level revenue structure
- order-level delivery and feedback outcomes

In [51]:
portfolio_items = fact_order_items.merge(
    fact_orders[["order_id", "is_late", "review_score", "is_low_review"]],
    on="order_id",
    how="left"
)

print("portfolio_items shape:", portfolio_items.shape)
print("Missing is_late:", portfolio_items["is_late"].isna().sum())
print("Missing review_score:", portfolio_items["review_score"].isna().sum())

portfolio_items shape: (110197, 32)
Missing is_late: 0
Missing review_score: 827


### 9.2 Category portfolio view

In [52]:
category_portfolio = (
    portfolio_items
    .groupby("product_category_name_english", as_index=False)
    .agg(
        revenue=("total_revenue", "sum"),
        orders=("order_id", "nunique"),
        items_sold=("order_item_id", "count"),
        late_order_exposure_rate=("is_late", "mean"),
        avg_review_score=("review_score", "mean"),
        low_review_exposure_rate=("is_low_review", "mean")
    )
    .sort_values("revenue", ascending=False)
)

category_portfolio["late_order_exposure_rate"] *= 100
category_portfolio["low_review_exposure_rate"] *= 100
category_portfolio["revenue_share_pct"] = (
    category_portfolio["revenue"] / category_portfolio["revenue"].sum()
) * 100

category_portfolio.head(10)

,product_category_name_english,revenue,orders,items_sold,late_order_exposure_rate,avg_review_score,low_review_exposure_rate,revenue_share_pct
43,health_beauty,1412089.53,8647,9465,7.564712,4.189729,12.371896,9.280038
70,watches_gifts,1264333.12,5495,5859,7.202594,4.071711,14.712408,8.309005
7,bed_bath_table,1225209.26,9272,10953,7.030037,3.923968,18.004200,8.051889
65,sports_leisure,1118256.91,7530,8431,6.310046,4.165493,12.952200,7.349015
15,computers_accessories,1032723.77,6530,7644,6.488749,3.986462,17.019885,6.786904
39,furniture_decor,880329.92,6307,8160,7.034314,3.953775,17.855392,5.785394
49,housewares,758392.25,5743,6795,5.003679,4.107904,14.113319,4.984039
20,cool_stuff,691680.89,3559,3718,5.836471,4.194904,11.780527,4.545622
5,auto,669454.75,3810,4140,7.028986,4.116004,13.768116,4.399555
42,garden_tools,567145.68,3448,4268,6.560450,4.084316,14.831303,3.727195


### 9.3 Seller portfolio view

In [53]:
seller_portfolio = (
    portfolio_items
    .groupby("seller_id", as_index=False)
    .agg(
        revenue=("total_revenue", "sum"),
        orders=("order_id", "nunique"),
        items_sold=("order_item_id", "count"),
        late_order_exposure_rate=("is_late", "mean"),
        avg_review_score=("review_score", "mean"),
        low_review_exposure_rate=("is_low_review", "mean")
    )
    .sort_values("revenue", ascending=False)
)

seller_portfolio["late_order_exposure_rate"] *= 100
seller_portfolio["low_review_exposure_rate"] *= 100
seller_portfolio["revenue_share_pct"] = (
    seller_portfolio["revenue"] / seller_portfolio["revenue"].sum()
) * 100

seller_portfolio.head(10)

,seller_id,revenue,orders,items_sold,late_order_exposure_rate,avg_review_score,low_review_exposure_rate,revenue_share_pct
834,4869f7a5dfa277a7dca6462dcf3b52b2,247007.06,1124,1148,10.540070,4.139474,13.327526,1.601885
1480,7c67e1448b00f6e969d365cea6b010ab,237806.69,973,1355,8.856089,3.347439,29.298893,1.542219
858,4a3ca9315b744ce9f8e9374361493884,231220.43,1772,1949,9.697281,3.828230,18.881478,1.499506
982,53243585a1d6dc2643021fd1853d8905,230797.02,348,400,3.000000,4.128141,11.000000,1.496760
2903,fa1c13f2614d7b5c4749cbc52fecda94,200833.50,578,579,9.153713,4.373913,9.499136,1.302441
2543,da8622b14eb17ae2831f4ac5b9dab84a,184706.78,1311,1548,6.395349,4.067769,14.922481,1.197857
1504,7e93a43ef30c4f03f38b393420bc753a,171973.55,319,322,4.658385,4.364486,8.074534,1.115279
188,1025f0e2d44d7041d6cf58b6550e0bfa,171924.96,910,1420,6.690141,3.869318,19.436620,1.114964
1450,7a67c85e85bb2ce8582c35f2203ad736,160278.52,1145,1155,5.194805,4.268091,10.043290,1.039435
1758,955fee9216a65b617aa5c0531780ce60,156606.48,1261,1472,6.453804,4.088235,13.315217,1.015621


### 9.4 Category segment logic

Categories are segmented using two dimensions:

- **commercial weight**, measured through revenue share
- **delivery-friction exposure**, measured through the share of category-linked items associated with late orders

Review-based feedback is then used as an additional reading layer, not as a primary segmentation axis.

In [54]:
category_revenue_threshold = category_portfolio["revenue_share_pct"].median()
category_exposure_threshold = category_portfolio["late_order_exposure_rate"].median()

print("Category revenue threshold (median):", round(category_revenue_threshold, 4))
print("Category exposure threshold (median):", round(category_exposure_threshold, 4))

Category revenue threshold (median): 0.3561
Category exposure threshold (median): 5.877


In [55]:
category_portfolio["category_segment_label"] = np.select(
    [
        (category_portfolio["revenue_share_pct"] >= category_revenue_threshold) &
        (category_portfolio["late_order_exposure_rate"] >= category_exposure_threshold),

        (category_portfolio["revenue_share_pct"] >= category_revenue_threshold) &
        (category_portfolio["late_order_exposure_rate"] < category_exposure_threshold),

        (category_portfolio["revenue_share_pct"] < category_revenue_threshold) &
        (category_portfolio["late_order_exposure_rate"] >= category_exposure_threshold),

        (category_portfolio["revenue_share_pct"] < category_revenue_threshold) &
        (category_portfolio["late_order_exposure_rate"] < category_exposure_threshold)
    ],
    [
        "Core Fragile Category",
        "Core Healthy Category",
        "Secondary Fragile Category",
        "Secondary Healthy Category"
    ],
    default="Unclassified"
)

### 9.5 Category segment summary

In [56]:
category_segment_summary = (
    category_portfolio
    .groupby("category_segment_label", as_index=False)
    .agg(
        categories=("product_category_name_english", "nunique"),
        revenue=("revenue", "sum")
    )
    .sort_values("revenue", ascending=False)
)

category_segment_summary["revenue_share_pct"] = (
    category_segment_summary["revenue"] / category_segment_summary["revenue"].sum()
) * 100

category_segment_summary

,category_segment_label,categories,revenue,revenue_share_pct
0,Core Fragile Category,23,11723011.93,77.041853
1,Core Healthy Category,13,2917083.49,19.170630
3,Secondary Healthy Category,22,325330.52,2.138023
2,Secondary Fragile Category,13,250993.97,1.649494


### 9.6 Category segment feedback overlay

In [57]:
category_segment_feedback = (
    category_portfolio
    .groupby("category_segment_label", as_index=False)
    .agg(
        avg_review_score=("avg_review_score", "mean"),
        low_review_exposure_rate=("low_review_exposure_rate", "mean"),
        late_order_exposure_rate=("late_order_exposure_rate", "mean")
    )
)

category_segment_feedback

,category_segment_label,avg_review_score,low_review_exposure_rate,late_order_exposure_rate
0,Core Fragile Category,4.068001,14.911751,7.165959
1,Core Healthy Category,4.124701,13.450732,4.670562
2,Secondary Fragile Category,4.082382,14.357706,8.242629
3,Secondary Healthy Category,4.132422,13.574358,3.220037


### 9.7 Seller segment logic

Sellers are segmented using the same conceptual two-dimensional logic:

- **commercial weight**, measured through revenue share
- **delivery-friction exposure**, measured through the share of seller-linked items associated with late orders

However, because the seller portfolio is much more fragmented than the category portfolio, the exposure distribution is highly skewed and contains many zero values. For that reason, the seller exposure threshold is set using a **higher percentile** rather than the median, in order to preserve a meaningful separation between healthier and more fragile seller groups.

Review-based feedback is again used as an additional interpretive layer.

In [63]:
seller_revenue_threshold = seller_portfolio["revenue_share_pct"].median()
seller_exposure_threshold = seller_portfolio["late_order_exposure_rate"].quantile(0.75)

print("Seller revenue threshold (median):", round(seller_revenue_threshold, 4))
print("Seller exposure threshold (75th percentile):", round(seller_exposure_threshold, 4))

Seller revenue threshold (median): 0.0067
Seller exposure threshold (75th percentile): 7.4074


In [64]:
seller_portfolio["seller_segment_label"] = np.select(
    [
        (seller_portfolio["revenue_share_pct"] >= seller_revenue_threshold) &
        (seller_portfolio["late_order_exposure_rate"] >= seller_exposure_threshold),

        (seller_portfolio["revenue_share_pct"] >= seller_revenue_threshold) &
        (seller_portfolio["late_order_exposure_rate"] < seller_exposure_threshold),

        (seller_portfolio["revenue_share_pct"] < seller_revenue_threshold) &
        (seller_portfolio["late_order_exposure_rate"] >= seller_exposure_threshold),

        (seller_portfolio["revenue_share_pct"] < seller_revenue_threshold) &
        (seller_portfolio["late_order_exposure_rate"] < seller_exposure_threshold)
    ],
    [
        "Core Fragile Seller",
        "Core Healthy Seller",
        "Secondary Fragile Seller",
        "Secondary Healthy Seller"
    ],
    default="Unclassified"
)

### 9.8 Seller segment summary

In [65]:
seller_segment_summary = (
    seller_portfolio
    .groupby("seller_segment_label", as_index=False)
    .agg(
        sellers=("seller_id", "nunique"),
        revenue=("revenue", "sum")
    )
    .sort_values("revenue", ascending=False)
)

seller_segment_summary["revenue_share_pct"] = (
    seller_segment_summary["revenue"] / seller_segment_summary["revenue"].sum()
) * 100

seller_segment_summary

,seller_segment_label,sellers,revenue,revenue_share_pct
1,Core Healthy Seller,983,9419445.97,61.086798
0,Core Fragile Seller,502,5451959.01,35.356933
3,Secondary Healthy Seller,1238,429210.13,2.783505
2,Secondary Fragile Seller,247,119158.64,0.772765


### 9.9 Seller segment feedback overlay

In [66]:
seller_segment_feedback = (
    seller_portfolio
    .groupby("seller_segment_label", as_index=False)
    .agg(
        avg_review_score=("avg_review_score", "mean"),
        low_review_exposure_rate=("low_review_exposure_rate", "mean"),
        late_order_exposure_rate=("late_order_exposure_rate", "mean")
    )
)

seller_segment_feedback

,seller_segment_label,avg_review_score,low_review_exposure_rate,late_order_exposure_rate
0,Core Fragile Seller,3.888148,19.596250,16.084079
1,Core Healthy Seller,4.229127,11.623423,2.247420
2,Secondary Fragile Seller,3.448698,30.262506,40.780058
3,Secondary Healthy Seller,4.333480,9.070175,0.082919


### Interpretation of the portfolio insights

The portfolio analysis shows that commercial exposure is not distributed uniformly across categories and sellers.

At the **category level**, the portfolio appears heavily concentrated in **core fragile categories**, which account for roughly **77%** of total revenue. By contrast, core healthy categories represent a much smaller share of the revenue base. This is an important result: the category mix is not only commercially concentrated, but also skewed toward segments more exposed to late-order risk.

The review overlay reinforces this reading. Core fragile categories are associated with both a **lower average review score** and a **higher low-review exposure rate** than core healthy categories. This suggests that the most commercially important parts of the category portfolio are also linked to weaker operational and customer-feedback outcomes.

At the **seller level**, the picture is more balanced. Most revenue is generated by **core healthy sellers**, who account for roughly **61%** of total revenue, while core fragile sellers still represent a substantial share of about **35%**. This indicates that seller concentration exists, but the seller portfolio is less skewed toward fragility than the category portfolio.

The seller feedback overlay points in the same direction. Core fragile sellers are associated with:
- weaker average review scores
- higher low-review exposure
- much higher late-order exposure

whereas core healthy sellers display materially stronger profiles on all three dimensions.

Overall, the portfolio view suggests that the business is not uniformly fragile across its commercial structure. Instead, fragility is concentrated in selected parts of the portfolio, especially on the **category side**, while the seller portfolio remains more mixed and contains a stronger healthy core.

This strengthens the broader project conclusion: revenue protection should not only focus on geography, but also on the commercially dominant portfolio components that are more exposed to weaker execution and weaker observed feedback.

## 10. Strategic Synthesis

The analysis conducted in this notebook leads to four main conclusions.

### 10.1 Revenue is more geographically concentrated than customer-concentrated

The commercial structure is driven primarily by a limited number of destination markets, especially **SP**, **RJ**, and **MG**. By contrast, customer-level concentration exists but remains much weaker, and the business relies on a very broad base of mostly single-order customers.

### 10.2 Operational friction is selective, not systemic

At the global level, delivery execution appears broadly stable. However, friction is not evenly distributed across markets. Some states show materially higher lateness than the global average, while others remain comparatively healthier.

### 10.3 Weaker execution is associated with weaker observed customer feedback

Late orders are associated with much lower review scores and much higher low-review rates than on-time orders. This does not establish causality by itself, but it clearly indicates that delivery friction is commercially relevant beyond pure logistics.

### 10.4 Exposure and fragility overlap in specific markets and portfolio components

The most important strategic issue is not broad underperformance, but the overlap between:
- high commercial exposure
- elevated delivery friction
- weaker feedback signals

This overlap appears most clearly in **RJ** and **BA** at the market level, and in commercially dominant but more fragile parts of the category portfolio.

These results support a prioritization logic focused on **commercially material execution risk**, rather than on treating all weak areas equally.

## 11. Strategic Recommendations

Based on the evidence developed in this notebook, the following strategic priorities emerge.

### 11.1 Prioritize fragile core markets first

Operational remediation should focus first on markets where commercial weight and execution fragility overlap most clearly.

This applies primarily to:
- **RJ**, which combines strong revenue contribution with weaker delivery and feedback outcomes
- **BA**, which shows a similarly fragile pattern despite a smaller revenue base

Secondary fragile markets such as **ES** and **SC** also deserve attention, but with lower priority than RJ.

### 11.2 Protect healthy commercial anchors

Not all large markets are fragile. **SP** stands out as a healthy commercial anchor, combining dominant revenue contribution with stronger-than-average execution and feedback. **MG**, **RS**, and **PR** also appear relatively healthier.

These markets should be protected operationally, since they stabilize the commercial base.

### 11.3 Monitor fragile portfolio components, especially on the category side

The category portfolio is more skewed toward fragility than the seller portfolio. This means that selected commercially dominant categories deserve closer monitoring, not because they can be blamed causally for all delivery issues, but because they are more exposed to weaker execution and weaker feedback signals.

### 11.4 Move from descriptive intelligence to predictive monitoring

The next logical extension of the project is to shift from descriptive execution analysis to **predictive late-delivery modelling**, so that risk can be identified before order completion rather than only diagnosed afterward.

## 12. Notebook Conclusion

This notebook transformed the analytical datasets into a structured revenue intelligence reading of the Olist business.

The analysis showed that:
- revenue is strongly concentrated geographically
- the customer base is broad but weakly recurrent
- delivery friction is selective rather than systemic
- weaker execution is associated with weaker review-based feedback
- the most important risks are located where commercial exposure and fragility overlap

These results provide the analytical basis for:
- dashboard storytelling
- strategic recommendations
- and the next project extension toward predictive late-delivery modeling.